In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['figure.figsize'] = 12,6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ####################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder

# 학습 모델 성능 관련 ####################################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
# 학습곡선
from sklearn.model_selection import learning_curve
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# 모델 성능평가 #############################################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
# 분류용
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

# 피처 선택 ################################################
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance

# 학습모델 ##################################################
#분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB

#회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import BayesianRidge

# 결정트리를 시각화할 수 있는 라이브러리
from sklearn.tree import plot_tree

# 차원축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# 연관규칙 학습
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# 불필요한 경고 뜨지 않게
import warnings
warnings.filterwarnings('ignore')

### Apriori

In [4]:
# 가상의 장바구니 데이터
dataset = [
    ['우유','빵','기저귀'],
    ['빵','기저귀','맥주','달걀'],
    ['우유','기저귀','맥주','콜라'],
    ['우유','빵','기저귀','맥주'],
    ['우유','빵','콜라']
]

In [11]:
# 데이터 전처리 (One-Hot Encoding 방식)
# Apriori 알고리즘은 True/False 형태의 행렬 데이터를 입력으로 받는다.
te = TransactionEncoder()
te.fit(dataset)
te_ary = te.transform(dataset)
df = pd.DataFrame(te_ary, columns=te.columns_)
df

,기저귀,달걀,맥주,빵,우유,콜라
0,True,False,False,True,True,False
1,True,True,True,True,False,False
2,True,False,True,False,True,True
3,True,False,True,True,True,False
4,False,False,False,True,True,True


In [12]:
# 빈번 항목 집합 찾기 (Support 지표 활용)
# min_support : 최소 지지도를 0.4(40%)로 설정 (전체 거래 중 40% 이상 등장한 항목만 추출)
frequent_itemsets = apriori(df, min_support=0.4, use_colnames=True)
frequent_itemsets

,support,itemsets
0,0.8,frozenset({기저귀})
1,0.6,frozenset({맥주})
2,0.8,frozenset({빵})
3,0.8,frozenset({우유})
4,0.4,frozenset({콜라})
5,0.6,"frozenset({맥주, 기저귀})"
6,0.6,"frozenset({빵, 기저귀})"
7,0.6,"frozenset({우유, 기저귀})"
8,0.4,"frozenset({맥주, 빵})"
9,0.4,"frozenset({맥주, 우유})"


In [14]:
# 연관 규칙 생성(Lift 지표 활용)
# metric : 평가지표 (lift, confidence 등)
# min_threshold : 최소 기준값
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({맥주}),frozenset({기저귀}),0.6,0.8,0.6,1.000000,1.250000,1.0,0.12,inf,0.500000,0.75,1.000000,0.875000
1,frozenset({기저귀}),frozenset({맥주}),0.8,0.6,0.6,0.750000,1.250000,1.0,0.12,1.6,1.000000,0.75,0.375000,0.875000
2,frozenset({우유}),frozenset({콜라}),0.8,0.4,0.4,0.500000,1.250000,1.0,0.08,1.2,1.000000,0.50,0.166667,0.750000
3,frozenset({콜라}),frozenset({우유}),0.4,0.8,0.4,1.000000,1.250000,1.0,0.08,inf,0.333333,0.50,1.000000,0.750000
4,"frozenset({맥주, 빵})",frozenset({기저귀}),0.4,0.8,0.4,1.000000,1.250000,1.0,0.08,inf,0.333333,0.50,1.000000,0.750000
5,"frozenset({빵, 기저귀})",frozenset({맥주}),0.6,0.6,0.4,0.666667,1.111111,1.0,0.04,1.2,0.250000,0.50,0.166667,0.666667
6,frozenset({맥주}),"frozenset({빵, 기저귀})",0.6,0.6,0.4,0.666667,1.111111,1.0,0.04,1.2,0.250000,0.50,0.166667,0.666667
7,frozenset({기저귀}),"frozenset({맥주, 빵})",0.8,0.4,0.4,0.500000,1.250000,1.0,0.08,1.2,1.000000,0.50,0.166667,0.750000
8,"frozenset({맥주, 우유})",frozenset({기저귀}),0.4,0.8,0.4,1.000000,1.250000,1.0,0.08,inf,0.333333,0.50,1.000000,0.750000
9,"frozenset({우유, 기저귀})",frozenset({맥주}),0.6,0.6,0.4,0.666667,1.111111,1.0,0.04,1.2,0.250000,0.50,0.166667,0.666667


In [24]:
# 필요한 것만 추출하여 결과를 확인한다.
# 조건 결과 지지도 신뢰도 향상도
result = rules[['antecedents','consequents','support','confidence','lift']]
# frozenset 지우기
result['antecedents'] = result['antecedents'].apply(lambda x : ', '.join(list(x)))
result['consequents'] = result['consequents'].apply(lambda x : ', '.join(list(x)))
result.sort_values(by='lift', ascending=False)

,antecedents,consequents,support,confidence,lift
0,맥주,기저귀,0.6,1.000000,1.250000
2,우유,콜라,0.4,0.500000,1.250000
3,콜라,우유,0.4,1.000000,1.250000
4,"맥주, 빵",기저귀,0.4,1.000000,1.250000
11,기저귀,"맥주, 우유",0.4,0.500000,1.250000
7,기저귀,"맥주, 빵",0.4,0.500000,1.250000
8,"맥주, 우유",기저귀,0.4,1.000000,1.250000
1,기저귀,맥주,0.6,0.750000,1.250000
6,맥주,"빵, 기저귀",0.4,0.666667,1.111111
5,"빵, 기저귀",맥주,0.4,0.666667,1.111111


### FP-growth

In [26]:
# 빈번 항목 집합 찾기(FP-Growth 알고리즘 사용)
# Apriori 대신 fpgrowth를 사용한다(속도만 개선한 것)
frequent_itemsets = fpgrowth(df, min_support=0.4, use_colnames=True)

In [27]:
# 연관 규칙 생성
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)

In [28]:
# 필요한 것만 추출하여 결과를 확인한다.
# 조건 결과 지지도 신뢰도 향상도
result = rules[['antecedents','consequents','support','confidence','lift']]
# frozenset 지우기
result['antecedents'] = result['antecedents'].apply(lambda x : ', '.join(list(x)))
result['consequents'] = result['consequents'].apply(lambda x : ', '.join(list(x)))
result.sort_values(by='lift', ascending=False)

,antecedents,consequents,support,confidence,lift
0,맥주,기저귀,0.6,1.000000,1.250000
2,"맥주, 빵",기저귀,0.4,1.000000,1.250000
6,"맥주, 우유",기저귀,0.4,1.000000,1.250000
5,기저귀,"맥주, 빵",0.4,0.500000,1.250000
11,콜라,우유,0.4,1.000000,1.250000
9,기저귀,"맥주, 우유",0.4,0.500000,1.250000
10,우유,콜라,0.4,0.500000,1.250000
1,기저귀,맥주,0.6,0.750000,1.250000
3,"빵, 기저귀",맥주,0.4,0.666667,1.111111
4,맥주,"빵, 기저귀",0.4,0.666667,1.111111
